# TREVL Demo Notebook

Interactive data visualization with TREVL YAML and Highcharts.

In [ ]:
$LOAD_PATH.unshift(File.expand_path('../lib', __dir__))
require 'trevl'
require 'trevl/notebook'

puts "TREVL v#{Trevl::VERSION} loaded"

## 1. Simple Bar Chart

In [ ]:
nb = Trevl::Notebook.new

salary_data = {
  "salary" => {
    "data" => [
      {"label" => "Junior",    "value" => 42000},
      {"label" => "Mid-Level", "value" => 58000},
      {"label" => "Senior",    "value" => 78000},
      {"label" => "Lead",      "value" => 92000},
      {"label" => "Principal", "value" => 110000}
    ]
  }
}

nb.chart(<<~YAML, data: salary_data)
  components:
  - id: salary_bar
    type: chart
    api: static
    highchartsData:
      chart:
        type: bar
      title:
        text: Salary by Seniority (EUR)
      colors: ["#003F85"]
      yAxis:
        title:
          text: Annual Salary (EUR)
      series:
      - name: Salary
        data:
          x: "$salary.data.label"
          y: "$salary.data.value"
YAML

## 2. Column Chart with Computed Fields

In [ ]:
skills_data = {
  "skills" => {
    "data" => [
      {"name" => "Python",     "demand" => 85},
      {"name" => "JavaScript", "demand" => 78},
      {"name" => "SQL",        "demand" => 72},
      {"name" => "Java",       "demand" => 65},
      {"name" => "Ruby",       "demand" => 45},
      {"name" => "Go",         "demand" => 38},
      {"name" => "Rust",       "demand" => 22}
    ]
  }
}

yaml = <<~YAML
  components:
  - id: skills_chart
    type: chart
    api: static
    computed:
    - name: color
      arguments:
        d: "$skills.data.demand"
      code: 'd > 70 ? "#003F85" : d > 50 ? "#4A90D9" : "#B0C4DE"'
    highchartsData:
      chart:
        type: column
      title:
        text: Programming Language Demand
      yAxis:
        title:
          text: Demand Score
        max: 100
      plotOptions:
        column:
          borderRadius: 4
          colorByPoint: true
      series:
      - name: Demand
        data:
          x: "$skills.data.name"
          y: "$skills.data.demand"
          color: "$color"
YAML

nb.chart(yaml, data: skills_data)

## 3. Postprocess: Top-N with Sorting

In [ ]:
cities_data = {
  "cities" => {
    "data" => [
      {"city" => "Berlin",     "vacancies" => 12500},
      {"city" => "Munich",     "vacancies" => 9800},
      {"city" => "Hamburg",    "vacancies" => 7200},
      {"city" => "Frankfurt",  "vacancies" => 6500},
      {"city" => "Cologne",    "vacancies" => 4800},
      {"city" => "Stuttgart",  "vacancies" => 4200},
      {"city" => "Dusseldorf", "vacancies" => 3900},
      {"city" => "Leipzig",    "vacancies" => 2100}
    ]
  }
}

nb.chart(<<~YAML, data: cities_data)
  components:
  - id: top_cities
    type: chart
    api: static
    postprocess: |
      $result = $result
        .sort(function(a, b) { return b.vacancies - a.vacancies; })
        .slice(0, 5);
    highchartsData:
      chart:
        type: bar
      title:
        text: Top 5 Cities by Job Vacancies
      colors: ["#E87722"]
      series:
      - name: Vacancies
        data:
          x: "$cities.data.city"
          y: "$cities.data.vacancies"
YAML

## 4. Score Component

In [ ]:
kpi_data = {
  "kpi" => {
    "data" => [{"avg_salary" => 62100}],
    "meta" => {"median" => 58000}
  }
}

nb.score(<<~YAML, data: kpi_data)
  components:
  - id: median_score
    type: score
    api: static
    display:
      value: "$kpi.data.avg_salary"
      unit: " EUR"
      header:
        title: Average Salary
YAML

## 5. Templates: Reusable Chart Styles

In [ ]:
Trevl.template_store.register("branded_bar", {
  "highchartsData" => {
    "chart" => {"type" => "bar"},
    "colors" => ["#003F85", "#E87722"],
    "plotOptions" => {
      "bar" => {"borderRadius" => 6, "groupPadding" => 0.1}
    },
    "legend" => {"enabled" => true},
    "credits" => {"enabled" => false}
  }
})

compare_data = {
  "compare" => {
    "data" => [
      {"region" => "North", "current" => 55000, "previous" => 52000},
      {"region" => "South", "current" => 48000, "previous" => 47500},
      {"region" => "East",  "current" => 43000, "previous" => 41000},
      {"region" => "West",  "current" => 51000, "previous" => 50000}
    ]
  }
}

nb.chart(<<~YAML, data: compare_data)
  components:
  - id: region_compare
    type: chart
    api: static
    template: branded_bar
    highchartsData:
      title:
        text: Salary Comparison by Region
      series:
      - name: Current Year
        data:
          x: "$compare.data.region"
          y: "$compare.data.current"
      - name: Previous Year
        data:
          x: "$compare.data.region"
          y: "$compare.data.previous"
YAML